# 움직임 시험 1 — 거친 폭발 블로킹 → LoRA 그림→그림(img2img)으로 장마다 다시 그리기 (Kaggle T4, 학습 없음)
목적: AI가 정한 거친 움직임(덩어리 크기·위치·타이밍)을 컬러 LoRA(2차 3바퀴)가 나카무라 화풍으로 장마다 다시 그릴 수 있는지, 장 사이 깜빡임이 얼마나인지 본다.
- 입력: 서버에서 만든 블로킹(움직임만 잡은 밑그림) 15장 — 나카무라 폭발 실측 타이밍(길이 15장, 정점 6장째, 화면 22.5%, 1·2·3장 걸이 섞음, 정점 뒤 4번 갈라짐, 파편 속도 반감 3장; tools/fx_blocking.py). 걸이 장은 같은 그림이라 같은 결과가 나옴.
- 강도(원래 그림을 얼마나 바꿀지) 0.4 · 0.55 · 0.7, 모든 장 같은 시드(1234)·같은 글.
- 설정: **GPU T4 x2**, **Internet On** → **Save Version → Save & Run All (Commit)**. 약 30–40분.
- 끝나면 Output의 `motion1_result.zip`(나란히 영상 `motion1.mp4` + 장별 그림)을 Drive 공유 링크로 알려 준다.


In [ ]:
# 1) 학습 환경: 서버에서 불러오기까지 확인한 조합을 별도 환경(/tmp/env)에 그대로 깐다(Kaggle 기본 패키지와 섞이지 않게)
!pip -q install uv
!uv venv -q /tmp/env --python 3.10
!uv pip install -q --python /tmp/env/bin/python torch==2.4.1 torchvision==0.19.1 --index-url https://download.pytorch.org/whl/cu124
!git clone -q --depth 1 --branch v0.9.1 https://github.com/kohya-ss/sd-scripts.git /tmp/sd-scripts
!cd /tmp/sd-scripts && uv pip install -q --python /tmp/env/bin/python -r requirements.txt
!uv pip install -q --python /tmp/env/bin/python lycoris-lora==3.1.1 "numpy==1.26.4"
!cd /tmp/sd-scripts && /tmp/env/bin/python -c "import torch, library, lycoris, bitsandbytes, cv2; print('torch', torch.__version__, 'cuda', torch.cuda.is_available())"

In [ ]:
# 2) 기본 모델(Hugging Face)과 2차 결과 zip(Drive 공유 링크)에서 컬러 LoRA 3바퀴만
!curl -sSL -C - -o /tmp/ilxl.safetensors https://huggingface.co/OnomaAIResearch/Illustrious-XL-v1.0/resolve/main/Illustrious-XL-v1.0.safetensors
!curl -sSL -o /tmp/v2.zip "https://drive.usercontent.google.com/download?id=1VjfgquMobDcMNxbwqc-0akpbcQ9BycqN&export=download&confirm=t"
!cd /tmp && unzip -q -o /tmp/v2.zip out/nkmr_fx_color-000003.safetensors && rm /tmp/v2.zip
!ls -la /tmp/ilxl.safetensors /tmp/out/

In [ ]:
# 3) 블로킹 그림(Drive)과 강도별 그림→그림
import os, glob, subprocess
!curl -sSL -o /tmp/blocking.zip "https://drive.usercontent.google.com/download?id=1lLCSZbiHE0QUO6xSw_uIeWiyUoNjlgDx&export=download&confirm=t"
!mkdir -p /tmp/blocking && cd /tmp/blocking && unzip -q -o /tmp/blocking.zip && ls *.png | wc -l
P = "nkmr_fx, vivid colors, anime screencap, explosion, fire, embers, white core orange rim, layered cel shading, no outline, sharp tips, large effect, no humans, city, night, wide shot"
N = "lowres, worst quality, text, watermark"
n = len(glob.glob('/tmp/blocking/*.png'))
open('/tmp/m.txt', 'w').write("\n".join(f"{P} --n {N} --w 1344 --h 768 --s 28 --l 6 --d 1234" for _ in range(n)))
L = "/tmp/out/nkmr_fx_color-000003.safetensors"
for s in (0.4, 0.55, 0.7):
    out = f"/kaggle/working/m/s{s}"
    os.makedirs(out, exist_ok=True)
    os.system(f"cd /tmp/sd-scripts && /tmp/env/bin/python sdxl_gen_img.py --ckpt /tmp/ilxl.safetensors --outdir {out} --from_file /tmp/m.txt "
              f"--image_path /tmp/blocking --strength {s} --sequential_file_name --images_per_prompt 1 --fp16 --no_half_vae --sdpa --sampler euler_a "
              f"--network_module lycoris.kohya --network_weights {L} --network_mul 1.0 --network_merge > {out}/log.txt 2>&1")
    print(s, len(glob.glob(out + '/*.png')), 'images')


In [ ]:
# 4) 나란히 영상(블로킹 | 0.4 / 0.55 | 0.7), 15장(24fps)을 3번 반복 + 깜빡임 수치
import cv2, numpy as np
cols = {"blocking": sorted(glob.glob('/tmp/blocking/*.png'))}
for s in (0.4, 0.55, 0.7):
    cols[f"s{s}"] = sorted(glob.glob(f'/kaggle/working/m/s{s}/*.png'))
print({k: len(v) for k, v in cols.items()})
def rd(p): return cv2.resize(cv2.imread(p), (672, 384))
frames = []
for i in range(len(cols["blocking"])):
    ims = []
    for k in cols:
        im = rd(cols[k][i]) if i < len(cols[k]) else np.zeros((384, 672, 3), np.uint8)
        cv2.putText(im, k, (8, 26), 0, 0.8, (255, 255, 255), 2)
        ims.append(im)
    frames.append(np.vstack([np.hstack(ims[:2]), np.hstack(ims[2:])]))
vw = cv2.VideoWriter('/kaggle/working/motion1.mp4', cv2.VideoWriter_fourcc(*'mp4v'), 24, (1344, 768))
for _ in range(3):
    for f in frames:
        vw.write(f)
vw.release()
for k, v in cols.items():
    a = [cv2.imread(p).astype(np.float32) for p in v]
    print(k, '이웃 장 평균 차', round(float(np.mean([np.abs(a[i + 1] - a[i]).mean() for i in range(len(a) - 1)])), 1))
cv2.imwrite('/kaggle/working/motion1_sheet.jpg', np.vstack([np.hstack([cv2.resize(cv2.imread(p), (336, 192)) for p in cols[k]]) for k in cols]), [cv2.IMWRITE_JPEG_QUALITY, 85])
!cd /kaggle/working && zip -q -r motion1_result.zip m motion1.mp4 motion1_sheet.jpg && ls -la /kaggle/working
